<a href="https://colab.research.google.com/github/nikolakr7/learningMLsys/blob/main/02_naive_matmul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mini-project after GPUMODE lecture 2: naive matmul

Hand-written CUDA matmul, one thread per output cell, benchmarked
against PyTorch on a T4. A is 1200x800, B is 800x1000.
Average of 20 runs after 3 warm-up runs.

| | Time | GFLOPS |
| :-- | :-- | :-- |
| Mine (naive) | 5.280 ms | 364 |
| PyTorch | 0.592 ms | 3,240 |

About 9x slower than PyTorch. Each thread re-reads a full row of A
and column of B from main GPU memory, so the same 7 MB of input gets
read about 1000 times over (7.7 GB of reads in total).

# Example of matrix add kernel to reference and see differences

In [1]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void matrix_add_kernel(float *a, float *b, float *out, int height, int width) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < height && col < width) {
        int i = row * width + col;
        out[i] = a[i] + b[i];
    }
}

int main() {
    int height = 1000;
    int width = 2000;
    int n = height * width;
    size_t bytes = n * sizeof(float);

    float *h_a = (float *)malloc(bytes);
    float *h_b = (float *)malloc(bytes);
    float *h_out = (float *)malloc(bytes);
    for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = 2.0f * i;
    }

    float *d_a;
    float *d_b;
    float *d_out;
    cudaMalloc((void **)&d_a, bytes);
    cudaMalloc((void **)&d_b, bytes);
    cudaMalloc((void **)&d_out, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((width + 15) / 16, (height + 15) / 16);
    matrix_add_kernel<<<blocks, threads>>>(d_a, d_b, d_out, height, width);
    cudaDeviceSynchronize();

    cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost);

    int wrong = 0;
    for (int i = 0; i < n; i++) {
        if (h_out[i] != h_a[i] + h_b[i]) wrong++;
    }
    printf("wrong: %d\n", wrong);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_out);
    free(h_a);
    free(h_b);
    free(h_out);
    return 0;
}

Writing matrix_add.cu


In [2]:
!nvcc matrix_add.cu -o matrix_add && ./matrix_add

wrong: 0


# My naive matmul, timed to compare to pytorch

In [1]:
%%writefile matmul.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

double now_ms() {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1000000.0;
}

__global__ void matmul_kernel(float *a, float *b, float *out, int M_height, int N_width, int K) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < M_height && col < N_width) {
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
          int a_idx = row * K + k;
          int b_idx = k * N_width + col;
          sum += a[a_idx] * b[b_idx];
        }
        int out_idx = row * N_width + col;
        out[out_idx] = sum;
    }
}

int main() {
  int N = 1000;  // number of columns in b. becomes the number of columns in the output matrix.
  int K = 800;   // number of multiplication computations per thread. It's the number of columns in a and rows in b. there must be the same number of both.
  int M = 1200;  // number of rows in a. becomes the number of rows in the output matrix.

  int n_a = M * K;
  int n_b = K * N;
  int n_out = M * N;

  size_t bytes_a = n_a * sizeof(float);
  size_t bytes_b = n_b * sizeof(float);
  size_t bytes_out = n_out * sizeof(float);

  float *h_a = (float *)malloc(bytes_a);
  float *h_b = (float *)malloc(bytes_b);
  float *h_out = (float *)malloc(bytes_out);

  for (int i = 0; i < n_a; i++) {
        h_a[i] = i % 7;
    }
  for (int i = 0; i < n_b; i++) {
        h_b[i] = i % 5;
    }

  float *d_a;
  float *d_b;
  float *d_out;
  cudaMalloc((void **)&d_a, bytes_a);
  cudaMalloc((void **)&d_b, bytes_b);
  cudaMalloc((void **)&d_out, bytes_out);
  cudaMemcpy(d_a, h_a, bytes_a, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, bytes_b, cudaMemcpyHostToDevice);

  dim3 threads(16, 16);
  dim3 blocks((N + 15) / 16, (M + 15) / 16);

  // warm-up runs, not timed
  for (int r = 0; r < 3; r++) {
      matmul_kernel<<<blocks, threads>>>(d_a, d_b, d_out, M, N, K);
  }
  cudaDeviceSynchronize();
  // timed runs
  int runs = 20;
  double start = now_ms();
  for (int r = 0; r < runs; r++) {
      matmul_kernel<<<blocks, threads>>>(d_a, d_b, d_out, M, N, K);
  }
  cudaDeviceSynchronize();
  double end = now_ms();
  printf("avg per run: %.3f ms\n", (end - start) / runs);

  cudaMemcpy(h_out, d_out, bytes_out, cudaMemcpyDeviceToHost);

  int wrong = 0;
  for (int i = 0; i < n_out; i++) {
    float sum = 0.0f;
    int row = i / N;
    int col = i % N;
    for (int k = 0; k < K; k++) {
      int a_idx = row * K + k;
      int b_idx = k * N + col;
      sum += h_a[a_idx] * h_b[b_idx];
    }
    if (h_out[i] != sum) wrong++;
  }
  printf("wrong: %d\n", wrong);

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_out);
  free(h_a);
  free(h_b);
  free(h_out);
  return 0;
}


Writing matmul.cu


In [2]:
!nvcc matmul.cu -o matmul && ./matmul

avg per run: 5.280 ms
wrong: 0


# rewritten with tiny matrices to trace and see outputs (only 2x2 output matrix)

In [19]:
%%writefile matmul.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void matmul_kernel(float *a, float *b, float *out, int M_height, int N_width, int K) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < M_height && col < N_width) {
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
          int a_idx = row * K + k;
          int b_idx = k * N_width + col;
          sum += a[a_idx] * b[b_idx];
        }
        int out_idx = row * N_width + col;
        out[out_idx] = sum;
    }
}

int main() {
  int N = 2;  // number of columns in b. becomes the number of columns in the output matrix.
  int K = 3;   // number of multiplication computations per thread. It's the number of columns in a and rows in b. there must be the same number of both.
  int M = 2;  // number of rows in a. becomes the number of rows in the output matrix.

  int n_a = M * K;
  int n_b = K * N;
  int n_out = M * N;

  size_t bytes_a = n_a * sizeof(float);
  size_t bytes_b = n_b * sizeof(float);
  size_t bytes_out = n_out * sizeof(float);

  float *h_a = (float *)malloc(bytes_a);
  float *h_b = (float *)malloc(bytes_b);
  float *h_out = (float *)malloc(bytes_out);

  for (int i = 0; i < n_a; i++) {
        h_a[i] = i + 1.0f;
    }
  for (int i = 0; i < n_b; i++) {
        h_b[i] = i + 7.0f;
    }

  float *d_a;
  float *d_b;
  float *d_out;
  cudaMalloc((void **)&d_a, bytes_a);
  cudaMalloc((void **)&d_b, bytes_b);
  cudaMalloc((void **)&d_out, bytes_out);
  cudaMemcpy(d_a, h_a, bytes_a, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, bytes_b, cudaMemcpyHostToDevice);

  dim3 threads(16, 16);
  dim3 blocks((N + 15) / 16, (M + 15) / 16);

  matmul_kernel<<<blocks, threads>>>(d_a, d_b, d_out, M, N, K);
  cudaDeviceSynchronize();

  cudaMemcpy(h_out, d_out, bytes_out, cudaMemcpyDeviceToHost);

  int wrong = 0;
  for (int i = 0; i < n_out; i++) {
    float sum = 0.0f;
    int row = i / N;
    int col = i % N;
    for (int k = 0; k < K; k++) {
      int a_idx = row * K + k;
      int b_idx = k * N + col;
      sum += h_a[a_idx] * h_b[b_idx];
    }
    if (h_out[i] != sum) wrong++;
    printf("%f\n", h_out[i]);
  }
  printf("wrong: %d\n", wrong);



  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_out);
  free(h_a);
  free(h_b);
  free(h_out);
  return 0;
}


Overwriting matmul.cu


In [20]:
!nvcc matmul.cu -o matmul && ./matmul

58.000000
64.000000
139.000000
154.000000
wrong: 0


# Pytorch matmul wow this is way simpler bless these engineers

In [3]:
import torch, time

M, K, N = 1200, 800, 1000

a = torch.rand(M, K, device="cuda")
b = torch.rand(K, N, device="cuda")
out = torch.empty(M, N, device="cuda")

# warm-up runs, not timed
for _ in range(3):
    torch.matmul(a, b, out=out)
torch.cuda.synchronize()

# timed runs
runs = 20
start = time.perf_counter()
for _ in range(runs):
    torch.matmul(a, b, out=out)
torch.cuda.synchronize()
end = time.perf_counter()
print(f"avg per run: {(end - start) / runs * 1000:.3f} ms")

avg per run: 0.592 ms


aaaand mines like 10x slower im cryin (gonna use tiling next)